# Efficiency Comparison: X-space vs Z-space Retrieval

This notebook 데이터 공간(X-space)과 잠재 공간(Z-space)에서의 retrieval 효율성을 비교합니다.

- **X-space (데이터 공간)**: `retrieve_X.py`에서 직접 데이터로 유사도 계산
- **Z-space (잠재 공간)**: `retrieve_Z.py`에서 embedding 모델을 사용하여 잠재 공간에서 유사도 계산

비교 항목:
1. 인덱스 빌드 시간
2. Query 인코딩 시간 (Z-space만 해당)
3. 검색 시간 (단일 쿼리, 배치 쿼리)
4. 메모리 사용량
5. 데이터베이스 크기

**주의**: Random initialized 모델을 사용하므로 실제 검색 품질은 고려하지 않고, 순수한 효율성만 비교합니다.


In [1]:
import os
import sys
import time
import math
import pickle
import psutil
import numpy as np
import pandas as pd
import torch
import faiss
from pathlib import Path
from tqdm import tqdm
from transformers import AutoConfig
from chronos import ChronosPipeline

# Add cross-rag to path
ROOT = Path('/home/seunghan.lee/workspace/cross-rag')
sys.path.insert(0, str(ROOT))

from retrieve_X import RetrieverX, load_database as load_database_X, pairwise_distance
from retrieve_Z import Retriever, load_database as load_database_Z, create_database as create_database_Z
import retrieve_Z  # Import module for monkey patching
from utils.tools import get_borders

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda:1


## 1. Random Initialized Embedding Model 생성


In [2]:
# Create embedding model for Z-space retrieval
# Note: retrieve_Z.py uses ChronosPipeline (from chronos package) which has embed() method
# Use the same approach as zeroshot.py
def create_embedding_model():
    """Create ChronosPipeline model with embed method for Z-space retrieval"""
    # Use same approach as zeroshot.py - this should work correctly
    pipeline = ChronosPipeline.from_pretrained(
        "amazon/chronos-t5-base",
        device_map=str(device),  # Use device_map as in zeroshot.py
        torch_dtype=torch.bfloat16,
    )
    return pipeline

embedding_model = create_embedding_model()
print(f'Embedding model created: {type(embedding_model)}')
print(f'Has embed method: {hasattr(embedding_model, "embed")}')


Embedding model created: <class 'chronos.chronos.ChronosPipeline'>
Has embed method: True


## 2. 설정 및 데이터 준비


In [3]:
# Configuration
dataset_name = 'ETTh2'
root_dir = ROOT / '../datasets/ETT-small'
retrieval_database_dir = ROOT / '../retrieval_database'
context_length = 512
prediction_length = 64
top_k = 10
seed = 42

metadata = {
    'database_name': [dataset_name],
    'lookback_length': context_length,
    'frequency': 'hour',
}

# Load data
data_path = root_dir / f'{dataset_name}.csv'
df = pd.read_csv(data_path)
variable_name = df.columns[1]  # First variable
raw_data = df[variable_name].values.astype(np.float32)

print(f'Dataset: {dataset_name}')
print(f'Variable: {variable_name}')
print(f'Data length: {len(raw_data)}')
print(f'Context length: {context_length}')
print(f'Prediction length: {prediction_length}')


Dataset: ETTh2
Variable: HUFL
Data length: 17420
Context length: 512
Prediction length: 64


## 3. X-space Retrieval 효율성 측정


In [53]:
def measure_x_space_efficiency(raw_data, context_length, prediction_length, top_k, metadata, retrieval_database_dir, root_dir, metric='euclidean'):
    """X-space retrieval 효율성 측정"""
    results = {}
    
    # Get borders
    border1s, border2s = get_borders('ETTh2', context_length, len(raw_data))
    
    # 1. 인덱스 빌드 시간 측정
    print(f'\n[X-space] Building index with metric={metric}...')
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024 / 1024  # MB
    
    retriever = RetrieverX(
        database_dir=str(retrieval_database_dir),
        root_dir=str(root_dir),
        metadata=metadata,
        seed=seed,
        lookback_length=context_length,
        metric=metric,
    )
    
    start_time = time.time()
    retriever.build_index(
        y_length=prediction_length,
        variable_filter=[variable_name],
        begin=border1s[0],
        end=border2s[0],
    )
    build_time = time.time() - start_time
    mem_after = process.memory_info().rss / 1024 / 1024  # MB
    
    results['build_time'] = build_time
    results['build_memory_mb'] = mem_after - mem_before
    results['num_slices'] = len(retriever.slices)
    results['slice_size_mb'] = retriever.slices.nbytes / 1024 / 1024
    
    print(f'  Build time: {build_time:.4f}s')
    print(f'  Memory usage: {results["build_memory_mb"]:.2f} MB')
    print(f'  Number of slices: {results["num_slices"]}')
    print(f'  Slice data size: {results["slice_size_mb"]:.2f} MB')
    
    # 2. 단일 쿼리 검색 시간 측정
    print(f'\n[X-space] Measuring single query search time...')
    query = raw_data[:context_length].reshape(1, -1)
    
    times = []
    for _ in range(10):  # 10회 측정하여 평균
        start_time = time.time()
        distances, boundary_idx, timestamp_idx = retriever.search(query, top_k=top_k)
        times.append(time.time() - start_time)
    
    results['single_query_time'] = np.mean(times)
    results['single_query_std'] = np.std(times)
    print(f'  Single query time: {results["single_query_time"]:.6f}s (std: {results["single_query_std"]:.6f}s)')
    
    # 3. 배치 쿼리 검색 시간 측정
    print(f'\n[X-space] Measuring batch query search time...')
    batch_sizes = [1, 10, 50, 100, 256, 512]
    batch_results = {}
    
    for batch_size in batch_sizes:
        # Create batch queries
        num_queries = min(batch_size, len(raw_data) - context_length - prediction_length)
        query_batch = np.array([raw_data[i:i+context_length] for i in range(num_queries)], dtype=np.float32)
        
        times = []
        for _ in range(5):  # 5회 측정
            start_time = time.time()
            distances, boundary_idx, timestamp_idx = retriever.search(query_batch, top_k=top_k)
            times.append(time.time() - start_time)
        
        avg_time = np.mean(times)
        batch_results[batch_size] = {
            'time': avg_time,
            'time_per_query': avg_time / num_queries,
            'queries_per_sec': num_queries / avg_time,
        }
        print(f'  Batch size {batch_size}: {avg_time:.4f}s ({batch_results[batch_size]["queries_per_sec"]:.2f} queries/sec)')
    
    results['batch_results'] = batch_results
    
    return results, retriever

# Test with different metrics
x_space_results = {}
for metric in ['euclidean', 'cosine']:
    print(f'\n\n=== X-space ({metric}) ===')
    x_space_results[metric], _ = measure_x_space_efficiency(
        raw_data, context_length, prediction_length, top_k, metadata, 
        retrieval_database_dir, root_dir, metric=metric
    )




=== X-space (euclidean) ===

[X-space] Building index with metric=euclidean...
Build X-space index with database: ['ETTh2_hour_512.pkl']
load database: ETTh2_hour_512.pkl
  Build time: 0.1648s
  Memory usage: 0.00 MB
  Number of slices: 8640
  Slice data size: 16.88 MB

[X-space] Measuring single query search time...
  Single query time: 0.003446s (std: 0.000141s)

[X-space] Measuring batch query search time...
  Batch size 1: 0.0033s (301.87 queries/sec)
  Batch size 10: 0.0572s (174.75 queries/sec)
  Batch size 50: 0.2694s (185.59 queries/sec)
  Batch size 100: 0.5330s (187.60 queries/sec)
  Batch size 256: 1.3415s (190.83 queries/sec)
  Batch size 512: 2.6394s (193.98 queries/sec)


=== X-space (cosine) ===

[X-space] Building index with metric=cosine...
Build X-space index with database: ['ETTh2_hour_512.pkl']
load database: ETTh2_hour_512.pkl
  Build time: 0.1793s
  Memory usage: 0.00 MB
  Number of slices: 8640
  Slice data size: 16.88 MB

[X-space] Measuring single query searc

## 4. Z-space Retrieval 효율성 측정


In [4]:
def measure_z_space_efficiency(raw_data, context_length, prediction_length, top_k, metadata, 
                                retrieval_database_dir, root_dir, embedding_model, dimension=768):
    """Z-space retrieval 효율성 측정"""
    results = {}
    
    # Get embedding model device - handle device_map case
    # ChronosPipeline uses inner_model, ChronosBoltPipeline uses model
    if hasattr(embedding_model, 'inner_model'):
        # For ChronosPipeline, get device from the actual model parameters
        try:
            model_device = next(embedding_model.inner_model.parameters()).device
        except StopIteration:
            # If model uses device_map, try to get device from encoder
            if hasattr(embedding_model.inner_model, 'encoder'):
                model_device = next(embedding_model.inner_model.encoder.parameters()).device
            else:
                model_device = device
    elif hasattr(embedding_model, 'model'):
        model_device = next(embedding_model.model.parameters()).device
    else:
        model_device = device  # fallback
    print(f'Embedding model device: {model_device}')
    print(f'Target device: {device}')
    
    # Monkey patch create_database to handle device properly
    # Import retrieve_Z module for patching
    import retrieve_Z
    
    # Store original function
    original_create_database = retrieve_Z.create_database
    
    # Create patched function with device parameter captured from closure
    def patched_create_database(raw_data, timestamps, lookback_length, embedding_model, metadata):
        embeddings = []
        sliced_timestamps = []
        
        # batch embedding
        batch_size = 512
        num_batchs = (len(raw_data) - lookback_length + 1 + batch_size - 1) // batch_size
        
        # Get device from embedding_model
        if hasattr(embedding_model, 'inner_model'):
            embed_device = next(embedding_model.inner_model.parameters()).device
        elif hasattr(embedding_model, 'model'):
            embed_device = next(embedding_model.model.parameters()).device
        else:
            embed_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        for batch_idx in tqdm(range(num_batchs)):
            # get start_idx and end_idx
            start_idx = batch_idx * batch_size
            end_idx = min(len(raw_data) - lookback_length + 1, start_idx + batch_size)
            
            # get batch data - IMPORTANT: specify device here
            batch_slices = [raw_data[i:i + lookback_length] for i in range(start_idx, end_idx)]
            batch_slices = torch.tensor(batch_slices, device=embed_device, dtype=torch.float32)
            
            # embedding
            batch_embeddings, _ = embedding_model.embed(batch_slices)
            eos_embeddings = batch_embeddings[:, -1, :].float().cpu().numpy()
            
            # append to embeddings
            embeddings.extend(eos_embeddings)
            sliced_timestamps.extend(timestamps[start_idx + lookback_length - 1:end_idx + lookback_length - 1])
        
        embeddings = np.array(embeddings)
        sliced_timestamps = np.array(sliced_timestamps)
        
        database = {
            'raw_data': raw_data,
            'timestamps': sliced_timestamps,
            'embeddings': embeddings,
            'metadata': metadata
        }
        return database
    
    # Patch the function in the module
    retrieve_Z.create_database = patched_create_database
    
    # Get borders
    border1s, border2s = get_borders('ETTh2', context_length, len(raw_data))
    
    # 1. 인덱스 빌드 시간 측정 (embedding 생성 포함)
    print(f'\n[Z-space] Building index (including embedding generation)...')
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / 1024 / 1024  # MB
    
    retriever = Retriever(
        database_dir=str(retrieval_database_dir),
        root_dir=str(root_dir),
        metadata=metadata,
        seed=seed,
        dimension=dimension,
        embedding_model=embedding_model,
        embedding_tuning=None,
    )
    
    start_time = time.time()
    retriever.build_index(
        y_length=prediction_length,
        variable_filter=[variable_name],
        begin=border1s[0],
        end=border2s[0],
    )
    build_time = time.time() - start_time
    mem_after = process.memory_info().rss / 1024 / 1024  # MB
    
    results['build_time'] = build_time
    results['build_memory_mb'] = mem_after - mem_before
    results['num_embeddings'] = retriever.index.ntotal
    results['embedding_size_mb'] = retriever.index.ntotal * dimension * 4 / 1024 / 1024  # float32 = 4 bytes
    
    print(f'  Build time: {build_time:.4f}s')
    print(f'  Memory usage: {results["build_memory_mb"]:.2f} MB')
    print(f'  Number of embeddings: {results["num_embeddings"]}')
    print(f'  Embedding data size: {results["embedding_size_mb"]:.2f} MB')
    
    # 2. Query 인코딩 시간 측정
    print(f'\n[Z-space] Measuring query encoding time...')
    query_sequence = raw_data[:context_length].reshape(1, -1)
    # Ensure query_tensor is on the same device as the model
    query_tensor = torch.tensor(query_sequence, dtype=torch.float32, device=model_device)
    # Warmup
#    print('embedding_model',embedding_model.device)
    print('query_tensor',query_tensor.device)
    with torch.no_grad():
        _ = embedding_model.embed(query_tensor)
    
    times = []
    for _ in range(10):  # 10회 측정
        start_time = time.time()
        with torch.no_grad():
            query_vector, _ = embedding_model.embed(query_tensor)
            query_vector = query_vector[:, -1, :].squeeze().cpu().float().numpy()
        times.append(time.time() - start_time)
    
    results['encoding_time'] = np.mean(times)
    results['encoding_std'] = np.std(times)
    print(f'  Encoding time: {results["encoding_time"]:.6f}s (std: {results["encoding_std"]:.6f}s)')
    
    # 3. 단일 쿼리 검색 시간 측정 (인코딩 포함)
    print(f'\n[Z-space] Measuring single query search time (with encoding)...')
    times_total = []
    times_search_only = []
    
    for _ in range(10):  # 10회 측정
        start_time = time.time()
        with torch.no_grad():
            query_vector, _ = embedding_model.embed(query_tensor)
            query_vector = query_vector[:, -1, :].squeeze().cpu().float().numpy().reshape(1, -1)
        encoding_time = time.time() - start_time
        
        start_time = time.time()
        distances, boundary_idx, timestamp_idx = retriever.search(query_vector, top_k=top_k)
        search_time = time.time() - start_time
        
        times_total.append(encoding_time + search_time)
        times_search_only.append(search_time)
    
    results['single_query_time_total'] = np.mean(times_total)
    results['single_query_time_search'] = np.mean(times_search_only)
    results['single_query_std'] = np.std(times_total)
    print(f'  Single query time (total): {results["single_query_time_total"]:.6f}s (std: {results["single_query_std"]:.6f}s)')
    print(f'  Single query time (search only): {results["single_query_time_search"]:.6f}s')
    
    # 4. 배치 쿼리 검색 시간 측정 (인코딩 포함)
    print(f'\n[Z-space] Measuring batch query search time (with encoding)...')
    batch_sizes = [1, 10, 50, 100, 256, 512]
    batch_results = {}
    
    for batch_size in batch_sizes:
        # Create batch queries
        num_queries = min(batch_size, len(raw_data) - context_length - prediction_length)
        query_batch = np.array([raw_data[i:i+context_length] for i in range(num_queries)], dtype=np.float32)
        query_tensor_batch = torch.tensor(query_batch, dtype=torch.float32).to(model_device)
        
        times_total = []
        times_search_only = []
        
        for _ in range(3):  # 3회 측정
            start_time = time.time()
            with torch.no_grad():
                query_vectors, _ = embedding_model.embed(query_tensor_batch)
                query_vectors = query_vectors[:, -1, :].squeeze().cpu().float().numpy()
            encoding_time = time.time() - start_time
            
            start_time = time.time()
            distances, boundary_idx, timestamp_idx = retriever.search(query_vectors, top_k=top_k)
            search_time = time.time() - start_time
            
            times_total.append(encoding_time + search_time)
            times_search_only.append(search_time)
        
        avg_time_total = np.mean(times_total)
        avg_time_search = np.mean(times_search_only)
        
        batch_results[batch_size] = {
            'time_total': avg_time_total,
            'time_search_only': avg_time_search,
            'time_per_query': avg_time_total / num_queries,
            'queries_per_sec': num_queries / avg_time_total,
        }
        print(f'  Batch size {batch_size}: {avg_time_total:.4f}s total ({batch_results[batch_size]["queries_per_sec"]:.2f} queries/sec)')
        print(f'    - Encoding: {avg_time_total - avg_time_search:.4f}s, Search: {avg_time_search:.4f}s')
    
    results['batch_results'] = batch_results
    
    # Restore original function
    retrieve_Z.create_database = original_create_database
    
    return results, retriever

print(f'\n\n=== Z-space ===')
z_space_results, _ = measure_z_space_efficiency(
    raw_data, context_length, prediction_length, top_k, metadata, 
    retrieval_database_dir, root_dir, embedding_model, dimension=768
)




=== Z-space ===
Embedding model device: cuda:1
Target device: cuda:1

[Z-space] Building index (including embedding generation)...
Build index with database: ['ETTh2_hour_512_Z.pkl']
load database: ETTh2_hour_512_Z.pkl
  Build time: 0.2989s
  Memory usage: 32.00 MB
  Number of embeddings: 8640
  Embedding data size: 25.31 MB

[Z-space] Measuring query encoding time...
query_tensor cuda:1


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cpu! (when checking argument for argument boundaries in method wrapper_CUDA_Tensor_bucketize)

In [ ]:
def get_database_size(database_path):
    """데이터베이스 파일 크기 반환 (MB)"""
    if os.path.exists(database_path):
        return os.path.getsize(database_path) / 1024 / 1024
    return None

db_sizes = {}

# X-space database
x_db_path = retrieval_database_dir / f'{dataset_name}_hour_{context_length}.pkl'
x_db_size = get_database_size(x_db_path)
if x_db_size:
    db_sizes['X-space'] = x_db_size
    print(f'X-space database size: {x_db_size:.2f} MB')
else:
    print(f'X-space database not found at {x_db_path}')

# Z-space database
z_db_path = retrieval_database_dir / f'{dataset_name}_hour_{context_length}_Z.pkl'
z_db_size = get_database_size(z_db_path)
if z_db_size:
    db_sizes['Z-space'] = z_db_size
    print(f'Z-space database size: {z_db_size:.2f} MB')
else:
    print(f'Z-space database not found at {z_db_path}')

if len(db_sizes) == 2:
    ratio = z_db_size / x_db_size
    print(f'\nSize ratio (Z/X): {ratio:.2f}x')


## 6. 결과 요약 및 비교


In [ ]:
print('\n' + '='*80)
print('EFFICIENCY COMPARISON SUMMARY')
print('='*80)

# 1. 인덱스 빌드 시간
print('\n1. Index Build Time:')
for metric, result in x_space_results.items():
    print(f'   X-space ({metric}): {result["build_time"]:.4f}s')
print(f'   Z-space: {z_space_results["build_time"]:.4f}s')

# 2. 메모리 사용량
print('\n2. Memory Usage (Index Building):')
for metric, result in x_space_results.items():
    print(f'   X-space ({metric}): {result["build_memory_mb"]:.2f} MB')
print(f'   Z-space: {z_space_results["build_memory_mb"]:.2f} MB')

# 3. 데이터 크기
print('\n3. Index Data Size:')
for metric, result in x_space_results.items():
    print(f'   X-space ({metric}): {result["slice_size_mb"]:.2f} MB ({result["num_slices"]} slices)')
print(f'   Z-space: {z_space_results["embedding_size_mb"]:.2f} MB ({z_space_results["num_embeddings"]} embeddings)')

# 4. 단일 쿼리 검색 시간
print('\n4. Single Query Search Time:')
for metric, result in x_space_results.items():
    print(f'   X-space ({metric}): {result["single_query_time"]:.6f}s')
print(f'   Z-space (total): {z_space_results["single_query_time_total"]:.6f}s')
print(f'   Z-space (search only): {z_space_results["single_query_time_search"]:.6f}s')
print(f'   Z-space (encoding only): {z_space_results["encoding_time"]:.6f}s')

# 5. 배치 쿼리 처리량
print('\n5. Batch Query Throughput (queries/sec):')
batch_sizes = [1, 10, 50, 100, 256, 512]
print(f'   Batch Size | X-space (euclidean) | X-space (cosine) | Z-space (total) | Z-space (search only)')
print(f'   ' + '-'*80)
for bs in batch_sizes:
    if bs in x_space_results['euclidean']['batch_results']:
        x_euc = x_space_results['euclidean']['batch_results'][bs]['queries_per_sec']
        x_cos = x_space_results['cosine']['batch_results'][bs]['queries_per_sec']
        z_total = z_space_results['batch_results'][bs]['queries_per_sec']
        # Recalculate for search only
        if 'time_search_only' in z_space_results['batch_results'][bs]:
            z_search = bs / z_space_results['batch_results'][bs]['time_search_only']
        else:
            z_search = 0
        print(f'   {bs:10d} | {x_euc:19.2f} | {x_cos:16.2f} | {z_total:15.2f} | {z_search:20.2f}')

# 6. 데이터베이스 파일 크기
if len(db_sizes) == 2:
    print('\n6. Database File Size:')
    print(f'   X-space: {db_sizes["X-space"]:.2f} MB')
    print(f'   Z-space: {db_sizes["Z-space"]:.2f} MB')
    print(f'   Ratio (Z/X): {db_sizes["Z-space"] / db_sizes["X-space"]:.2f}x')

print('\n' + '='*80)


## 7. visualization


In [ ]:
import matplotlib.pyplot as plt

# 1. 배치 크기별 처리량 비교
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Throughput (queries/sec)
ax = axes[0]
batch_sizes = [1, 10, 50, 100, 256, 512]
valid_batch_sizes = [bs for bs in batch_sizes if bs in x_space_results['euclidean']['batch_results']]

x_euc_throughput = [x_space_results['euclidean']['batch_results'][bs]['queries_per_sec'] for bs in valid_batch_sizes]
x_cos_throughput = [x_space_results['cosine']['batch_results'][bs]['queries_per_sec'] for bs in valid_batch_sizes]
z_total_throughput = [z_space_results['batch_results'][bs]['queries_per_sec'] for bs in valid_batch_sizes]
z_search_throughput = [bs / z_space_results['batch_results'][bs]['time_search_only'] for bs in valid_batch_sizes]

ax.plot(valid_batch_sizes, x_euc_throughput, 'o-', label='X-space (euclidean)', linewidth=2)
ax.plot(valid_batch_sizes, x_cos_throughput, 's-', label='X-space (cosine)', linewidth=2)
ax.plot(valid_batch_sizes, z_total_throughput, '^-', label='Z-space (total)', linewidth=2)
ax.plot(valid_batch_sizes, z_search_throughput, 'v--', label='Z-space (search only)', linewidth=2, alpha=0.7)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Throughput (queries/sec)', fontsize=12)
ax.set_title('Batch Query Throughput Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

# Query time per query
ax = axes[1]
x_euc_time = [x_space_results['euclidean']['batch_results'][bs]['time_per_query'] * 1000 for bs in valid_batch_sizes]
x_cos_time = [x_space_results['cosine']['batch_results'][bs]['time_per_query'] * 1000 for bs in valid_batch_sizes]
z_total_time = [z_space_results['batch_results'][bs]['time_per_query'] * 1000 for bs in valid_batch_sizes]
z_search_time = [z_space_results['batch_results'][bs]['time_search_only'] / bs * 1000 for bs in valid_batch_sizes]

ax.plot(valid_batch_sizes, x_euc_time, 'o-', label='X-space (euclidean)', linewidth=2)
ax.plot(valid_batch_sizes, x_cos_time, 's-', label='X-space (cosine)', linewidth=2)
ax.plot(valid_batch_sizes, z_total_time, '^-', label='Z-space (total)', linewidth=2)
ax.plot(valid_batch_sizes, z_search_time, 'v--', label='Z-space (search only)', linewidth=2, alpha=0.7)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Time per Query (ms)', fontsize=12)
ax.set_title('Query Latency Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout()
os.makedirs('experiments/figures', exist_ok=True)
plt.savefig('experiments/figures/efficiency_comparison_throughput.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. 메모리 및 데이터 크기 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Build time
ax = axes[0]
methods = ['X-space\n(euclidean)', 'X-space\n(cosine)', 'Z-space']
build_times = [x_space_results['euclidean']['build_time'], x_space_results['cosine']['build_time'], z_space_results['build_time']]
bars = ax.bar(methods, build_times, color=['#3498db', '#3498db', '#e74c3c'], alpha=0.7)
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_title('Index Build Time', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, (bar, time) in enumerate(zip(bars, build_times)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{time:.2f}s', 
            ha='center', va='bottom', fontsize=10)

# Memory usage
ax = axes[1]
memory_usage = [x_space_results['euclidean']['build_memory_mb'], x_space_results['cosine']['build_memory_mb'], z_space_results['build_memory_mb']]
bars = ax.bar(methods, memory_usage, color=['#3498db', '#3498db', '#e74c3c'], alpha=0.7)
ax.set_ylabel('Memory (MB)', fontsize=12)
ax.set_title('Memory Usage (Index Building)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, (bar, mem) in enumerate(zip(bars, memory_usage)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{mem:.1f}MB', 
            ha='center', va='bottom', fontsize=10)

# Data size
ax = axes[2]
data_sizes = [x_space_results['euclidean']['slice_size_mb'], x_space_results['cosine']['slice_size_mb'], z_space_results['embedding_size_mb']]
bars = ax.bar(methods, data_sizes, color=['#3498db', '#3498db', '#e74c3c'], alpha=0.7)
ax.set_ylabel('Size (MB)', fontsize=12)
ax.set_title('Index Data Size', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for i, (bar, size) in enumerate(zip(bars, data_sizes)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{size:.1f}MB', 
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('experiments/figures/efficiency_comparison_resources.png', dpi=300, bbox_inches='tight')
plt.show()


## 8. 결과 분석 및 결론


In [ ]:
print('\n' + '='*80)
print('KEY FINDINGS')
print('='*80)

# Build time ratio
build_time_ratio = z_space_results['build_time'] / x_space_results['euclidean']['build_time']
print(f'\n1. Index Build Time:')
print(f'   - Z-space는 X-space보다 {build_time_ratio:.2f}x 느립니다.')
print(f'   - 이유: Z-space는 embedding 생성이 필요하므로 추가 시간이 소요됩니다.')

# Memory ratio
memory_ratio = z_space_results['build_memory_mb'] / x_space_results['euclidean']['build_memory_mb']
print(f'\n2. Memory Usage:')
print(f'   - Z-space는 X-space보다 {memory_ratio:.2f}x 많은 메모리를 사용합니다.')
print(f'   - 이유: Embedding 모델과 인덱스가 메모리에 상주합니다.')

# Data size ratio
data_size_ratio = z_space_results['embedding_size_mb'] / x_space_results['euclidean']['slice_size_mb']
print(f'\n3. Index Data Size:')
print(f'   - Z-space 인덱스는 X-space보다 {data_size_ratio:.2f}x 큽니다.')
print(f'   - 이유: 각 slice를 768차원 embedding으로 변환합니다.')

# Query time comparison
query_time_ratio_total = z_space_results['single_query_time_total'] / x_space_results['euclidean']['single_query_time']
query_time_ratio_search = z_space_results['single_query_time_search'] / x_space_results['euclidean']['single_query_time']
print(f'\n4. Single Query Search Time:')
print(f'   - Z-space (total)는 X-space보다 {query_time_ratio_total:.2f}x 느립니다.')
print(f'   - Z-space (search only)는 X-space보다 {query_time_ratio_search:.2f}x 느립니다.')
print(f'   - Z-space의 인코딩 시간: {z_space_results["encoding_time"]*1000:.2f}ms')
print(f'   - Z-space의 검색 시간: {z_space_results["single_query_time_search"]*1000:.2f}ms')

# Batch throughput
if 256 in z_space_results['batch_results']:
    throughput_256_z = z_space_results['batch_results'][256]['queries_per_sec']
    throughput_256_x = x_space_results['euclidean']['batch_results'][256]['queries_per_sec']
    throughput_ratio = throughput_256_x / throughput_256_z
    print(f'\n5. Batch Throughput (batch_size=256):')
    print(f'   - X-space: {throughput_256_x:.2f} queries/sec')
    print(f'   - Z-space: {throughput_256_z:.2f} queries/sec')
    print(f'   - X-space가 {throughput_ratio:.2f}x 빠릅니다.')

print(f'\n6. Trade-offs:')
print(f'   - X-space: 빠른 검색 속도, 적은 메모리, 간단한 구현')
print(f'   - Z-space: 더 품질 높은 검색 (학습된 모델 사용 시), 하지만 느린 속도와 많은 메모리')
print(f'   - 실제 사용 시에는 검색 품질과 효율성의 트레이드오프를 고려해야 합니다.')

print('\n' + '='*80)
